In [ ]:
!pip install pydub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp -r /content/drive/MyDrive/BanglaSpeechData /content/

In [ ]:
import numpy as np
import pandas as pd
import librosa
import soundfile
import glob,pickle
from sklearn.model_selection import train_test_split,cross_val_score
# from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import keras
from keras.layers import Dense, Dropout, Flatten
from keras.layers import Conv2D, MaxPooling2D
from keras.utils import to_categorical
from sklearn.svm import SVC

import os
from sklearn.preprocessing import normalize
from IPython.display import SVG
# from keras.utils.vis_utils import model_to_dot
import matplotlib.pyplot as plt
# from pydub import AudioSegment


from IPython.lib.display import Audio
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)
from mpl_toolkits.axes_grid1 import make_axes_locatable
import seaborn as sns

from pydub import AudioSegment
from pydub.playback import play

from IPython.display import clear_output, display

In [ ]:
Root = "/content/BanglaSpeechData/MyDataset";
# Root = "/content/drive/MyDrive/BanglaSpeechData";
# Root = "/content/drive/MyDrive/MyDataset/complete";
# Root = "/content/drive/MyDrive/BanglaSpeechData";
# Root = "/content/drive/MyDrive/BanglaSpeechData/Dataset";
# Root = "/content/drive/MyDrive/BanglaSpeechData";
os.chdir(Root)

# Emotions

In [ ]:
emotions={

  '01':'happy',
  '02':'sad',
  '03':'angry',
  '04':'surprised',
  '05':'neutral',
  '06':'disgust',
  '07':'fear',
}

#Emotions to observe
observed_emotions=['happy', 'sad', 'angry', 'surprised','neutral','disgust','fear']
# observed_emotions=['happy' ]

#Feature Extraction

In [ ]:
def show_plot(times,x):
  print('Nos. samples: '+str(len(x)))

  print('Duration: '+str(times[-1])+ 'seconds')
  fig = plt.subplots(figsize=(12,2))
  ax = plt.subplot(1,1,1)
  ax.plot(times, x)
  ax.grid(True)
  plt.ylabel('amplitude [in A.U.]', fontsize=14)
  plt.xlabel('time [in sec]', fontsize=14)
  plt.xticks(fontsize=13)
  plt.yticks(fontsize=13)
  ax.spines['right'].set_visible(False)
  ax.spines['top'].set_visible(False)
  plt.show()


###Final Extraction Function

### Trim silence at the beginning and end

In [ ]:
def extract_feature(file_name, mfcc, chroma, mel, tonnetz):
    with soundfile.SoundFile(file_name) as sound_file:
        X = sound_file.read(dtype="float32")
        sample_rate = sound_file.samplerate

        # Trim silence at the beginning and end
        # X, _ = librosa.effects.trim(X, top_db=20)

        result = np.array([])

        if mfcc:
            mfccs = np.mean(librosa.feature.mfcc(y=X, sr=sample_rate, n_mfcc=40).T, axis=0)
            result = np.hstack((result, mfccs))

        if chroma:
            stft = np.abs(librosa.stft(X))
            chroma = np.mean(librosa.feature.chroma_stft(S=stft, sr=sample_rate).T, axis=0)
            result = np.hstack((result, chroma))

        if mel:
            mel = np.mean(librosa.feature.melspectrogram(y=X, sr=sample_rate).T, axis=0)
            result = np.hstack((result, mel))

        if tonnetz:
            ton = np.mean(librosa.feature.tonnetz(y=X, sr=sample_rate).T, axis=0)
            result = np.hstack((result, ton))

    return result


In [ ]:
def load_data(link):
    times = 0
    x,y=[],[]
    # for file in glob.glob("/content/drive/MyDrive/BanglaSpeechData/*/*/*.wav"):
    for file in glob.glob(link):
    # for file in glob.glob("/content/drive/MyDrive/BanglaSpeechData/MyDataset/Actor*/*.wav"):
    # for file in glob.glob("/content/drive/MyDrive/BanglaSpeechData/*/*/*.wav"):
    # for file in glob.glob("/content/drive/MyDrive/BanglaSpeechData/Dataset/Actor */*.wav"):
    # for file in glob.glob("/content/drive/MyDrive/BanglaSpeechData/Dataset/Actor 01/03-01-01-01-01-01-01.wav"):
        file_name=os.path.basename(file)

        #converting stereo audio to mono
        sound = AudioSegment.from_wav(file)
        sound = sound.set_channels(1)
        sound.export(file, format="wav")


        emotion=emotions[file_name.split("-")[2]]


        if emotion not in observed_emotions:
            continue

        # print(f'currently loading {emotion} emotion.')


        feature=extract_feature(file, mfcc=True, chroma=False, mel=False,tonnetz=True)

        x.append(feature)
        y.append(file_name.split("-")[2])

        times += 1

        display_data = f'{times} data loaded ! and last Emotion was {emotion} '
        display(display_data, display_id='update_display')
        clear_output(wait=True)  # Clear the previous output

    return x,y

### Emotion Count Plot

# Load Data

In [ ]:
# Split the data into training and test sets

X,Y = load_data("/content/BanglaSpeechData/Dataset/Actor*/*.wav")
# X,Y = load_data("/content/drive/MyDrive/BanglaSpeechData/Dataset/Actor*/*.wav")


In [ ]:
# Split the data into training and test sets

X1,Y1 = load_data("/content/BanglaSpeechData/MyDataset/Actor*/*.wav")
# X1,Y1 = load_data("/content/drive/MyDrive//BanglaSpeechData/MyDataset/Actor*/*.wav")


In [ ]:
# Split the data into training and test sets

X2,Y2 = load_data("/content/BanglaSpeechData/Renamed/*/*.wav")
# X2,Y2 = load_data("/content/drive/MyDrive//BanglaSpeechData/Renamed/*/*.wav")


##Split train and test

###Preprossing data

In [ ]:
from sklearn.preprocessing import normalize

X = np.array(X)
Y = np.array(Y)
Y = Y.astype(int)

X1 = np.array(X1)
Y1 = np.array(Y1)
Y1 = Y1.astype(int)

X2 = np.array(X2)
Y2 = np.array(Y2)
Y2 = Y2.astype(int)

# Reshape X and X_test
X = X.reshape(X.shape[0], -1)

# Reshape X and X_test
X1 = X1.reshape(X1.shape[0], -1)

# Reshape X and X_test
# X2 = X2.reshape(X2.shape[0], -1)

# Apply normalization
X = normalize(X, axis=1, norm='l1')

# Apply normalization
X1 = normalize(X1, axis=1, norm='l1')

# Apply normalization
# X2 = normalize(X2, axis=1, norm='l1')





In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

def plot_emotion_count(y_type, title="Title"):
    emo_list = Counter(y_type)
    emotion_name = list(emo_list.keys())
    emotion_count = list(emo_list.values())

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(emotion_name, emotion_count, color='maroon', width=0.4)

    ax.set_title(title, fontsize=18)
    ax.set_xlabel("Emotion", fontsize=14)
    ax.set_ylabel("Count", fontsize=14)

    plt.show()

In [ ]:
print(X.max())
print(X1.max())
print(X2.max())


In [ ]:
# # X_train, X_test_val, y_train, y_test_val = train_test_split(X, Y, test_size=0.2, random_state=42)



X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
X_train1, X_test1, y_train1, y_test1 = train_test_split(X1, Y1, test_size=0.2, random_state=42)
X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, Y2, test_size=0.2, random_state=42)
# X_test,X_val,  y_test,y_val, = train_test_split(X_test_val, y_test_val, test_size=0.2)

plot_emotion_count(y_train,"Train")
plot_emotion_count(y_test,"Test")

plot_emotion_count(y_train1,"Train")
plot_emotion_count(y_test1,"Test")

plot_emotion_count(y_train2,"Train")
plot_emotion_count(y_test2,"Test")
# plot_emotion_count(y_val,"Validation")
# # plot_emotion_count(y_val,"Valid")

In [ ]:
import numpy as np

# Concatenate the training sets
X_train = np.concatenate((X_train, X_train1))
y_train = np.concatenate((y_train, y_train1))


In [ ]:
print(X_train.shape)
print(y_train.shape)

In [ ]:
while y_train.min() != 0:

  y_train = y_train - 1
  y_test = y_test - 1


# while y_train1.min() != 0:
#   y_train1 = y_train1 - 1
#   y_test1 = y_test1 - 1


while y_train2.min() != 0:
  y_train2 = y_train2 - 1
  y_test2 = y_test2 - 1
#   # y_val = y_val - 1



In [ ]:
# print(X_train.max())
# print(X_train.min())

# print(y_train.max())
# print(y_train.min())

# print(X_train2.shape)
# print(y_train2.shape)
num_classes = 7

In [ ]:
print(X_train.shape)

In [ ]:
# # Reshape X and X_test
# X = X.reshape(X.shape[0], -1)

# # Apply normalization
# X = normalize(X, axis=1, norm='l1')
X.shape

#This is Visual Area

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Sample label data
labels = y_train  # Replace with your actual label data

# Create a histogram
plt.figure(figsize=(8, 6))
plt.hist(labels, bins=np.arange(7) - 0.5, rwidth=0.8, alpha=0.7, align='mid')
plt.xticks(range(6))
plt.xlabel('Class Labels')
plt.ylabel('Frequency')
plt.title('Label Distribution')
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Get unique labels
unique_labels = np.unique(y_train)

# Create subplots for each class
plt.figure(figsize=(15, 10))
for i, label in enumerate(unique_labels):
    plt.subplot(3, 3, i+1)
    plt.hist(X_train[y_train == label].flatten(), bins=50, alpha=0.5, label=f'Class {label}')
    plt.title(f'Distribution of Features for Class {label}')
    plt.xlabel('Feature Value')
    plt.ylabel('Frequency')
    plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Assuming X_train and y_train are your data and labels

# Get unique labels
unique_labels = np.unique(y_train)

# Create a dictionary to store mean feature matrices for each class
class_mean_feature_matrices = {}

# Calculate mean feature matrix for each class
for label in unique_labels:
    # Get indices of samples with the current label
    indices = np.where(y_train == label)[0]

    # Extract features for samples with the current label
    features = X_train[indices]

    # Calculate mean of features along the samples
    mean_features = np.mean(features, axis=0)

    # Store the mean feature matrix in the dictionary
    class_mean_feature_matrices[label] = mean_features

# Plot heatmaps for each class
plt.figure(figsize=(10, 6))
for label, mean_features in class_mean_feature_matrices.items():
    plt.subplot(2, 4, label + 1)
    plt.imshow(mean_features.reshape(1, -1), cmap='viridis', aspect='auto')
    plt.title(f'Class {label}')
    plt.colorbar()

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming X_train and y_train are your data and labels

# Get unique labels
unique_labels = np.unique(y_train)

# Create a dictionary to store mean feature matrices for each class
class_mean_feature_matrices = {}

# Calculate mean feature matrix for each class
for label in unique_labels:
    # Get indices of samples with the current label
    indices = np.where(y_train == label)[0]

    # Extract features for samples with the current label
    features = X_train[indices]

    # Calculate mean of features along the samples
    # mean_features = np.mean(features, axis=0)

    mean_features = features;

    # Store the mean feature matrix in the dictionary
    class_mean_feature_matrices[label] = features



# Plot heatmaps for each class
plt.figure(figsize=(12, 8))
for label, mean_features in class_mean_feature_matrices.items():
    plt.figure(figsize=(10, 6))

    # Reshape mean_features to a 2D array for heatmap visualization
    mean_features_2d = mean_features.reshape(1, -1)

    sns.heatmap(mean_features_2d, cmap='viridis', annot=True)
    plt.title(f'Feature Heatmap for Class {label}')
    plt.xlabel('Feature Index')
    plt.ylabel('Sample')

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming X_train and y_train are your data and labels

# Get unique labels
unique_labels = np.unique(y_train)

# Create a dictionary to store mean feature matrices for each class
class_mean_feature_matrices = {}

# Calculate mean feature matrix for each class
for label in unique_labels:
    # Get indices of samples with the current label
    indices = np.where(y_train == label)[0]

    # Extract features for samples with the current label
    features = X_train[indices]

    # Calculate mean of features along the samples
    mean_features = np.mean(features, axis=0)

    # Store the mean feature matrix in the dictionary
    class_mean_feature_matrices[label] = mean_features

# Plot heatmaps for each class
for label, mean_features in class_mean_feature_matrices.items():
    plt.figure(figsize=(10, 6))

    # Reshape mean_features to a 2D array for heatmap visualization
    mean_features_2d = mean_features.reshape(1, -1)

    sns.heatmap(mean_features_2d, cmap='viridis', annot=True)
    plt.title(f'Feature Heatmap for Class {label}')
    plt.xlabel('Feature Index')
    plt.ylabel('Sample')

plt.show()


In [ ]:
print(X.shape)
print(Y.shape)
print(X.max())
print(X.min())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Assuming X_train represents your time steps or samples
# And y_train represents your corresponding amplitudes

# Create a range of time steps
time_steps = np.arange(len(y_train))

# Plot the data
plt.figure(figsize=(12, 2))
plt.plot(time_steps, y_train)
plt.grid(True)
plt.ylabel('amplitude [in A.U.]', fontsize=14)
plt.xlabel('time [in samples]', fontsize=14)
plt.xticks(fontsize=13)
plt.yticks(fontsize=13)
plt.show()


In [ ]:
fs = 32000
fname1 = '03-01-01-01-01-01-01.wav'
fname2 = '03-01-02-01-01-01-01.wav'
fname3 = '03-01-03-02-01-02-01.wav'

dname = '/content/drive/MyDrive/BanglaSpeechData/Dataset/Actor 01/'
# load
x1, sr1 = librosa.load(dname+fname1)
x2, sr2 = librosa.load(dname+fname2)
x3, sr3 = librosa.load(dname+fname3)
# x1 = x1/np.max(x1)
# x2 = x2/np.max(x2)
# x3 = x3/np.max(x3)
times1 = np.arange(len(x1))/sr1
times2 = np.arange(len(x2))/sr2
times3 = np.arange(len(x3))/sr3
# listen



plt.subplots(figsize=(12,2))
plt.grid(True)
plt.plot(np.arange(len(x1))/sr1, x1)
# plt.plot(np.arange(len(x2))/sr2, x2)
# plt.plot(np.arange(len(x3))/sr3, x3)

plt.show()



In [ ]:
fs = 32000
fname1 = '03-01-01-01-01-01-01.wav'
fname2 = '03-01-02-01-01-01-01.wav'
fname3 = '03-01-03-02-01-02-01.wav'
fname4 = '03-01-04-02-01-02-01.wav'
fname5 = '03-01-05-02-01-02-01.wav'

dname = '/content/drive/MyDrive/BanglaSpeechData/Dataset/Actor 03/'
# load
x1, sr1 = librosa.load(dname+fname1)
x2, sr2 = librosa.load(dname+fname2)
x3, sr3 = librosa.load(dname+fname3)
x4, sr4 = librosa.load(dname+fname4)
x5, sr5 = librosa.load(dname+fname5)

In [ ]:
# Audio(x1, rate=sr1, autoplay=True)
show_plot(times1,x1)
show_plot(times2,x2)
show_plot(times3,x3)
show_plot(times4,x4)
show_plot(times6,x6)

# Audio(x2, rate=sr2, autoplay=True)

# Audio(x3, rate=sr3, autoplay=True)




#Machine Learning Models

##Global data for compiler

In [ ]:
from keras.optimizers import RMSprop
from keras.callbacks import EarlyStopping

# Set the learning rate
learning_rate = 0.001

# Optionally, you can also set other hyperparameters like rho (decay factor) and epsilon
rho = 0.9
epsilon = 1e-08

# Compile the model with RMSprop optimizer
# optimizer = RMSprop(learning_rate=learning_rate, rho=rho, epsilon=epsilon)


early_stopping = EarlyStopping(monitor='sparse_categorical_accuracy', patience=20)
# early_stopping = EarlyStopping(monitor='accuracy', patience=10)

# optimizer=keras.optimizers.Adam(learning_rate=0.0001)

# metrics=[
#     keras.metrics.SparseCategoricalAccuracy()
# ]

# metrics=["accuracy"]

# Loss function
# loss = keras.losses.SparseCategoricalCrossentropy()
# loss = 'mean_squared_error'//////////
input_shape = (X_train.shape[1], 1)
num_classes = len(np.unique(y_train))

##Naive bias

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import cross_val_score


In [ ]:
new_X_train = X_train - X_train.min()
new_X_test = X_test - X_test.min()

In [ ]:
# Create a Multinomial Naive Bayes classifier
nb_classifier = MultinomialNB()

# Train the classifier on your training data
nb_classifier.fit(new_X_train, y_train)


In [ ]:
# Make predictions on the test data
y_pred = nb_classifier.predict(new_X_test)


In [ ]:
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')

# Generate a classification report
classification_rep = classification_report(y_test, y_pred)
print('Classification Report:\n', classification_rep)


In [ ]:
# Perform cross-validation
cv_scores = cross_val_score(nb_classifier, new_X_train, y_train, cv=5)  # 5-fold cross-validation
print('Cross-Validation Scores:', cv_scores)
print('Mean CV Accuracy:', cv_scores.mean())

## Generate a Model and then compile it

##NeuroFusion

In [ ]:
from keras.models import Sequential

def generalModel(input_shape, num_classes):
  model = Sequential()
  model.add(Dense(2048, input_dim=180, activation='relu'))
  model.add(Dense(1024, activation='relu'))
  model.add(Dense(512, activation='relu'))
  model.add(Dense(256, activation='relu'))
  model.add(Dense(128, activation='relu'))
  model.add(Dense(64, activation='relu'))
  model.add(Dense(32, activation='relu'))
  model.add(Dense(16, activation='relu'))
  model.add(Dense(8, activation='relu'))
  model.add(Dense(num_classes, activation='softmax'))
  return model


## Best Hyper Peremeterised Simple Mode

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Conv1D, MaxPooling1D, Flatten

# Best Hyper Peremeterised Simple Model

def generalModel_hp(hp):

  num_hidden_layers = 3
  num_units = 512
  dropout_rate = 0.3681083627739924
  learning_rate = 0.00001

  if hp:
    num_hidden_layers = hp.Choice('num_hidden_layers',values=[3,4,5])
    num_units = hp.Choice('num_units',values=[600,512,256,128])
    dropout_rate = hp.Choice('dropout_rate',values=[0.1,0.2,0.3])
    learning_rate = hp.Choice('learning_rate',values=[0.001,0.002,0.003])


  model = Sequential()
  model.add(Flatten(input_shape=(X_train.shape[1],1)))

  for _ in range(0,num_hidden_layers):
    model.add(Dense(num_units,activation='relu'))
    model.add(Dropout(dropout_rate))

  model.add(Dense(num_classes, activation='softmax'))

  model.compile(

      optimizer=keras.optimizers.Adam(learning_rate=0.0001),
      loss=loss,
      metrics=keras.metrics.SparseCategoricalAccuracy()
      )


  return model



In [ ]:
X_train.shape[1]

In [ ]:
model_with_hp = generalModel_hp(None)
model_with_hp.summary()

In [ ]:
!pip install -U keras-tuner

In [ ]:
import kerastuner

In [ ]:
class CustomTuner(kerastuner.tuners.BayesianOptimization):
  def run_trial(self,trial,*args,**kwargs):
    kwargs['batch_size'] = trial.hyperparameters.Int('batch_size',64,128,step=32)
    return super(CustomTuner,self).run_trial(trial,*args,**kwargs)



In [ ]:
tuner = CustomTuner(
    generalModel_hp,
    objective='val_sparse_categorical_accuracy',
    max_trials = 80,
    directory = 'logs',
    project_name = 'generalModel_hp',
    overwrite=True
)

In [ ]:
tuner.search_space_summary()

In [ ]:
from keras import callbacks
tuner.search(
    X_train,y_train,
    validation_data=(X_test,y_test),
    epochs=200,
    verbose=True,callbacks=[early_stopping]
)

In [ ]:
tuner.results_summary(1)

In [ ]:
model = tuner.get_best_models(num_models=1)[0]

In [ ]:
model.summary()

In [ ]:
best_model_fit_history = model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=200,batch_size=64,callbacks=[early_stopping])

## CovNetX

In [ ]:
from keras.models import Sequential
from keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

# Simple CNN Model

def CNN_model(input_shape, num_classes):
    model = Sequential()

    # First Convolutional layer
    model.add(Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))

    # Second Convolutional layer
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))

    # Third Convolutional layer
    model.add(Conv1D(filters=85, kernel_size=3, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))

    # Flatten the output from the previous layer
    model.add(Flatten())

    # Fully connected layers
    model.add(Dense(512, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(256, activation='relu'))
    model.add(Dropout(0.5))

    # Output layer with 'softmax' activation for multi-class classification
    model.add(Dense(num_classes, activation='softmax'))

    return model


## CovNetM1

In [ ]:
from keras.models import Sequential
from keras.layers import Conv1D, MaxPooling1D, Dense, Flatten, Dropout



def hp_CNN_model(input_shape, num_classes):


  model = Sequential()
  model.add(Dense(180,input_dim=180,input_shape=input_shape))

  # First convolutional layer
  model.add(Conv1D(filters=220, kernel_size=10, activation='relu'))
  model.add(MaxPooling1D(pool_size=2))
  model.add(Dropout(0.1))

  # Second convolutional layer
  model.add(Conv1D(filters=220, kernel_size=10, activation='relu'))
  model.add(MaxPooling1D(pool_size=10))
  model.add(Dropout(0.1))

  # Third convolutional layer
  model.add(Conv1D(filters=100, kernel_size=1, activation='relu'))
  model.add(MaxPooling1D(pool_size=2))
  model.add(Dropout(0.1))

  # Flatten the output from the previous layer
  model.add(Flatten())

  # Fully connected layers
  model.add(Dense(256, activation='relu'))
  model.add(Dropout(0.1))
  model.add(Dense(256, activation='relu'))
  model.add(Dropout(0.1))
  model.add(Dense(256, activation='relu'))
  model.add(Dropout(0.1))
  model.add(Dense(256, activation='relu'))

  # Output layer
  model.add(Dense(num_classes, activation='softmax'))

  return model



## CascadeCovM2

In [ ]:
from keras.models import Sequential
from keras.layers import Conv1D, MaxPooling1D, Dense, Flatten, Dropout
from keras.optimizers import Adam
from keras.losses import CategoricalCrossentropy
from keras.metrics import CategoricalAccuracy


def best_CNN_model(input_shape, num_classes):

  model = Sequential()
  model.add(Dense(180,input_dim=180,input_shape=input_shape))

  # First convolutional layer
  model.add(Conv1D(filters=180, kernel_size=3, activation='relu'))
  model.add(MaxPooling1D(pool_size=2))
  model.add(Dropout(0.2))

  # Second convolutional layer
  model.add(Conv1D(filters=180, kernel_size=3, activation='relu'))
  model.add(MaxPooling1D(pool_size=2))
  model.add(Dropout(0.2))

  # Third convolutional layer
  model.add(Conv1D(filters=360, kernel_size=3, activation='relu'))
  model.add(MaxPooling1D(pool_size=2))
  model.add(Dropout(0.2))

  # Flatten the output from the previous layer
  model.add(Flatten())

  # Fully connected layers
  model.add(Dense(720, activation='relu'))
  model.add(Dropout(0.3))
  model.add(Dense(360, activation='relu'))
  model.add(Dropout(0.3))
  model.add(Dense(180, activation='relu'))
  model.add(Dropout(0.3))
  model.add(Dense(90, activation='relu'))

  # Output layer
  model.add(Dense(num_classes, activation='softmax'))
  return model



## Deep3LSTM

In [ ]:
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from keras.optimizers import Adam
from keras.losses import CategoricalCrossentropy
from keras.metrics import CategoricalAccuracy

def LSTM_MODEL(input_shape, num_classes):
    model = Sequential()

    # LSTM layers
    model.add(LSTM(units=180, return_sequences=True, input_shape=input_shape))
    model.add(Dropout(0.2))

    model.add(LSTM(units=180, return_sequences=True))
    model.add(Dropout(0.2))

    model.add(LSTM(units=360, return_sequences=True))
    model.add(Dropout(0.2))

    # Flatten the output from the LSTM layer
    model.add(Flatten())

    # Fully connected layers
    model.add(Dense(720, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(360, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(180, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(90, activation='relu'))

    # Output layer
    model.add(Dense(num_classes, activation='softmax'))

    return model

##Double Layer CNN Model


In [ ]:
from keras.models import Sequential
from keras.layers import Conv1D, MaxPooling1D, Dense, Flatten

def general_cnn_model(input_shape, num_classes):
  model = Sequential()
  model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape))
  model.add(MaxPooling1D(pool_size=2))

  model.add(Conv1D(filters=256, kernel_size=3, activation='relu'))
  model.add(MaxPooling1D(pool_size=2))

  model.add(Flatten())

  model.add(Dense(512, activation='relu'))
  model.add(Dense(256, activation='relu'))
  model.add(Dense(128, activation='relu'))
  model.add(Dense(64, activation='relu'))
  model.add(Dense(num_classes, activation='softmax'))
  return model


##Double CNN model 2

In [ ]:
from keras.models import Sequential
from keras.layers import Conv1D, MaxPooling1D, Dense, Flatten

def CRNN_model(input_shape, num_classes):
  model = Sequential()
  model.add(Conv1D(filters=180, kernel_size=3, activation='relu', input_shape=input_shape))
  model.add(MaxPooling1D(pool_size=2))
  model.add(Conv1D(filters=180*3, kernel_size=(3, ), activation='relu'))
  model.add(MaxPooling1D(pool_size=2))
  model.add(Flatten())
  model.add(Dense(units=720, activation='relu'))
  model.add(Dense(units=360, activation='relu'))
  model.add(Dense(units=180, activation='relu'))
  model.add(Dense(units=60, activation='relu'))
  model.add(Dense(units=num_classes, activation='softmax'))
  return model

## Make a simple
LSTM model 2

In [ ]:
from keras.layers import Dense, LSTM, Dropout
from keras.models import Sequential
from keras.optimizers import Adam

# Assuming X_train, Y_train, X_test, and Y_test have the correct shapes

def general_LSTM_model(input_shape, num_classes):
  # Define the model
  model = Sequential()

  model.add(LSTM(180, return_sequences=True, input_shape=input_shape))
  model.add(Dropout(0.2))

  model.add(LSTM(180*3, return_sequences=True))
  model.add(Dropout(0.2))

  model.add(LSTM(256))
  model.add(Dropout(0.2))

  model.add(Dense(128, activation='relu'))
  model.add(Dropout(0.5))

  model.add(Dense(num_classes, activation='softmax'))
  return model


## CNN without flatten

In [ ]:
from keras.layers import Dense, Dropout, Conv1D, GlobalMaxPooling1D, MaxPooling1D
from keras.models import Sequential

def another_CNN_model(input_shape, num_classes):
  # Define the model
  model = Sequential()

  model.add(Conv1D(filters=180, kernel_size=3, activation='relu', input_shape=input_shape))
  model.add(MaxPooling1D(pool_size=2))

  model.add(Conv1D(filters=180*3, kernel_size=3, activation='relu'))
  model.add(MaxPooling1D(pool_size=2))

  model.add(Conv1D(filters=180*2, kernel_size=3, activation='relu'))
  model.add(GlobalMaxPooling1D())

  model.add(Dense(128, activation='relu'))
  model.add(Dropout(0.5))

  model.add(Dense(56, activation='relu'))
  model.add(Dropout(0.5))

  # model.add(Dense(y_train.shape[1], activation='softmax'))
  model.add(Dense(num_classes, activation='softmax'))
  return model


##SVM Model

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Assuming X_train, y, X_test, and Y_test have the correct shapes

# Create an SVM model
svm_model = SVC(kernel='linear', C=50,verbose=1)

# Train the model on the training data
svm_model.fit(X_train, y_train)

# Make predictions on the test data
y_pred = svm_model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")


##Data fit and accuracy calculating

In [ ]:
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)
print(X_train1.shape, y_train1.shape, X_test1.shape, y_test1.shape)
print(X_train2.shape, y_train2.shape, X_test2.shape, y_test2.shape)


#Compiler

In [ ]:
def model_compiler(my_model):
  # Compile the model with the specified optimizer, loss function, and metrics
  my_model.compile(
      optimizer=keras.optimizers.Adam(learning_rate=0.0001),
      loss=keras.losses.SparseCategoricalCrossentropy(),
      metrics=[keras.metrics.SparseCategoricalAccuracy()]
  )

  return my_model


In [ ]:
# # For BanglaSER only
input_shape = (X_train.shape[1], 1)
num_classes = len(np.unique(y_train))

ser_generalModel =  model_compiler(generalModel(input_shape, num_classes))
ser_CNN_model =  model_compiler(CNN_model(input_shape, num_classes))
ser_hp_CNN_model =  model_compiler(hp_CNN_model(input_shape, num_classes))
ser_best_CNN_model =  model_compiler(best_CNN_model(input_shape, num_classes))
ser_LSTM_MODEL =  model_compiler(LSTM_MODEL(input_shape, num_classes))
# ser_general_cnn_model =  model_compiler(general_cnn_model(input_shape, num_classes))
# ser_CRNN_model =  model_compiler(CRNN_model(input_shape, num_classes))
# ser_general_LSTM_model =  model_compiler(general_LSTM_model(input_shape, num_classes))
# ser_another_CNN_model =  model_compiler(another_CNN_model(input_shape, num_classes))

# # For mydataset only
# input_shape = (X_train1.shape[1], 1)
# num_classes = len(np.unique(y_train1))

# my_generalModel =  model_compiler(generalModel(input_shape, num_classes))
# my_CNN_model =  model_compiler(CNN_model(input_shape, num_classes))
# my_hp_CNN_model =  model_compiler(hp_CNN_model(input_shape, num_classes))
# my_best_CNN_model =  model_compiler(best_CNN_model(input_shape, num_classes))
# my_LSTM_MODEL =  model_compiler(LSTM_MODEL(input_shape, num_classes))
# my_general_cnn_model =  model_compiler(general_cnn_model(input_shape, num_classes))
# my_CRNN_model =  model_compiler(CRNN_model(input_shape, num_classes))
# my_general_LSTM_model =  model_compiler(general_LSTM_model(input_shape, num_classes))
# my_another_CNN_model =  model_compiler(another_CNN_model(input_shape, num_classes))

# # For sust only
# input_shape = (X_train2.shape[1], 1)
# num_classes = len(np.unique(y_train2))

# sust_generalModel =  model_compiler(generalModel(input_shape, num_classes))
# sust_CNN_model =  model_compiler(CNN_model(input_shape, num_classes))
# sust_hp_CNN_model =  model_compiler(hp_CNN_model(input_shape, num_classes))
# sust_best_CNN_model =  model_compiler(best_CNN_model(input_shape, num_classes))
# sust_LSTM_MODEL =  model_compiler(LSTM_MODEL(input_shape, num_classes))
# sust_general_cnn_model =  model_compiler(general_cnn_model(input_shape, num_classes))
# sust_CRNN_model =  model_compiler(CRNN_model(input_shape, num_classes))
# sust_general_LSTM_model =  model_compiler(general_LSTM_model(input_shape, num_classes))
# sust_another_CNN_model =  model_compiler(another_CNN_model(input_shape, num_classes))

#Data fit

In [ ]:
from tensorflow import expand_dims

# For BanglaSER only
ser_generalModel_history = ser_generalModel.fit(expand_dims(X_train, axis=-1),y_train,epochs=500, callbacks=[early_stopping])
ser_CNN_model_history = ser_CNN_model.fit(expand_dims(X_train, axis=-1),y_train,epochs=500, callbacks=[early_stopping])
ser_hp_CNN_model_history = ser_hp_CNN_model.fit(expand_dims(X_train, axis=-1),y_train,epochs=500, callbacks=[early_stopping])
ser_best_CNN_model_history = ser_best_CNN_model.fit(expand_dims(X_train, axis=-1),y_train,epochs=500, callbacks=[early_stopping])
# ser_LSTM_MODEL_history = ser_LSTM_MODEL.fit(expand_dims(X_train, axis=-1),y_train,epochs=500, callbacks=[early_stopping])
# ser_general_cnn_model_history = ser_general_cnn_model.fit(expand_dims(X_train, axis=-1),y_train,epochs=500, callbacks=[early_stopping])
# ser_CRNN_model_history = ser_CRNN_model.fit(expand_dims(X_train, axis=-1),y_train,epochs=500, callbacks=[early_stopping])
# ser_general_LSTM_model_history = ser_general_LSTM_model.fit(expand_dims(X_train, axis=-1),y_train,epochs=500, callbacks=[early_stopping])
# ser_another_CNN_model_history = ser_another_CNN_model.fit(expand_dims(X_train, axis=-1),y_train,epochs=500, callbacks=[early_stopping])


# # For mydataset only
# my_generalModel_history = my_generalModel.fit(expand_dims(X_train1, axis=-1),y_train1,epochs=500, callbacks=[early_stopping])
# my_CNN_model_history = my_CNN_model.fit(expand_dims(X_train1, axis=-1),y_train1,epochs=500, callbacks=[early_stopping])
# my_hp_CNN_model_history = my_hp_CNN_model.fit(expand_dims(X_train1, axis=-1),y_train1,epochs=500, callbacks=[early_stopping])
# my_best_CNN_model_history = my_best_CNN_model.fit(expand_dims(X_train1, axis=-1),y_train1,epochs=500, callbacks=[early_stopping])

# my_LSTM_MODEL_history = my_LSTM_MODEL.fit(expand_dims(X_train1, axis=-1),y_train1,epochs=500, callbacks=[early_stopping])
# my_general_cnn_model_history = my_general_cnn_model.fit(expand_dims(X_train1, axis=-1),y_train1,epochs=500, callbacks=[early_stopping])
# my_CRNN_model_history = my_CRNN_model.fit(expand_dims(X_train1, axis=-1),y_train1,epochs=500, callbacks=[early_stopping])
# my_general_LSTM_model_history = my_general_LSTM_model.fit(expand_dims(X_train1, axis=-1),y_train1,epochs=500, callbacks=[early_stopping])
# my_another_CNN_model_history = my_another_CNN_model.fit(expand_dims(X_train1, axis=-1),y_train1,epochs=500, callbacks=[early_stopping])

# For sust only
# sust_generalModel_history = sust_generalModel.fit(expand_dims(X_train2, axis=-1),y_train2,epochs=500, callbacks=[early_stopping])
# sust_CNN_model_history = sust_CNN_model.fit(expand_dims(X_train2, axis=-1),y_train2,epochs=500, callbacks=[early_stopping])
# sust_hp_CNN_model_history = sust_hp_CNN_model.fit(expand_dims(X_train2, axis=-1),y_train2,epochs=500, callbacks=[early_stopping])
# sust_best_CNN_model_history = sust_best_CNN_model.fit(expand_dims(X_train2, axis=-1),y_train2,epochs=500, callbacks=[early_stopping])
# sust_LSTM_MODEL_history = sust_LSTM_MODEL.fit(expand_dims(X_train2, axis=-1),y_train2,epochs=500, callbacks=[early_stopping])
# sust_general_cnn_model_history = sust_general_cnn_model.fit(expand_dims(X_train2, axis=-1),y_train2,epochs=500, callbacks=[early_stopping])
# sust_CRNN_model_history = sust_CRNN_model.fit(expand_dims(X_train2, axis=-1),y_train2,epochs=500, callbacks=[early_stopping])
# sust_general_LSTM_model_history = sust_general_LSTM_model.fit(expand_dims(X_train2, axis=-1),y_train2,epochs=500, callbacks=[early_stopping])
# sust_another_CNN_model_history = sust_another_CNN_model.fit(expand_dims(X_train2, axis=-1),y_train2,epochs=500, callbacks=[early_stopping])


In [ ]:
from keras.models import save_model
save_model(ser_generalModel, '/content/drive/MyDrive/Model/FinalModels/ser_ext_generalModel.h5')
save_model(ser_CNN_model, '/content/drive/MyDrive/Model/FinalModels/ser_ext_CNN_model.h5')
save_model(ser_hp_CNN_model, '/content/drive/MyDrive/Model/FinalModels/ser_ext_hp_CNN_model.h5')
save_model(ser_best_CNN_model, '/content/drive/MyDrive/Model/FinalModels/ser_ext_best_CNN_model.h5')
# save_model(ser_LSTM_MODEL, '/content/drive/MyDrive/Model/FinalModels/ser_LSTM_MODEL.h5')
# save_model(ser_general_cnn_model, '/content/drive/MyDrive/Model/FinalModels/ser_general_cnn_model.h5')
# save_model(ser_CRNN_model, '/content/drive/MyDrive/Model/FinalModels/ser_CRNN_model.h5')
# save_model(ser_general_LSTM_model, '/content/drive/MyDrive/Model/FinalModels/ser_general_LSTM_model.h5')
# save_model(ser_another_CNN_model, '/content/drive/MyDrive/Model/FinalModels/ser_another_CNN_model.h5')


## my dataset
# save_model(my_generalModel, '/content/drive/MyDrive/Model/FinalModels/my_generalModel.h5')
# save_model(my_CNN_model, '/content/drive/MyDrive/Model/FinalModels/my_CNN_model.h5')
# save_model(my_hp_CNN_model, '/content/drive/MyDrive/Model/FinalModels/my_hp_CNN_model.h5')
# save_model(my_best_CNN_model, '/content/drive/MyDrive/Model/FinalModels/my_best_CNN_model.h5')
# save_model(my_LSTM_MODEL, '/content/drive/MyDrive/Model/FinalModels/my_LSTM_MODEL.h5')
# save_model(my_general_cnn_model, '/content/drive/MyDrive/Model/FinalModels/my_general_cnn_model.h5')
# save_model(my_CRNN_model, '/content/drive/MyDrive/Model/FinalModels/my_CRNN_model.h5')
# save_model(my_general_LSTM_model, '/content/drive/MyDrive/Model/FinalModels/my_general_LSTM_model.h5')
# save_model(my_another_CNN_model, '/content/drive/MyDrive/Model/FinalModels/my_another_CNN_model.h5')


## for SUST
# save_model(sust_generalModel, '/content/drive/MyDrive/Model/FinalModels/sust_generalModel.h5')
# save_model(sust_CNN_model, '/content/drive/MyDrive/Model/FinalModels/sust_CNN_model.h5')
# save_model(sust_hp_CNN_model, '/content/drive/MyDrive/Model/FinalModels/sust_hp_CNN_model.h5')
# save_model(sust_best_CNN_model, '/content/drive/MyDrive/Model/FinalModels/sust_best_CNN_model.h5')
# save_model(sust_LSTM_MODEL, '/content/drive/MyDrive/Model/FinalModels/sust_LSTM_MODEL.h5')
# save_model(sust_general_cnn_model, '/content/drive/MyDrive/Model/FinalModels/sust_general_cnn_model.h5')
# save_model(sust_CRNN_model, '/content/drive/MyDrive/Model/FinalModels/sust_CRNN_model.h5')
# save_model(sust_general_LSTM_model, '/content/drive/MyDrive/Model/FinalModels/sust_general_LSTM_model.h5')
# save_model(sust_another_CNN_model, '/content/drive/MyDrive/Model/FinalModels/sust_another_CNN_model.h5')


###Model Summary

In [ ]:
SVG(model_to_dot(model).create(prog='dot',format='svg'))

##for accuracy

In [ ]:
from matplotlib import pyplot as plt
def print_history_acc(history,model_name):

  l = np.arange(len(history.history["loss"]))

  fig = plt.figure(dpi=150)
  plt.plot(l,history.history['sparse_categorical_accuracy'])
  # plt.plot(l,history.history['val_accuracy'])
  plt.title(f'model accuracy of {model_name}')
  plt.ylabel('accuracy')
  plt.xlabel('epoch')
  plt.legend(['sparse_categorical_accuracy','val_accuracy'])

  plt.savefig(f'/content/drive/MyDrive/Model/photos/accuracy_{model_name}')
  plt.show()

  plt.close(fig)  # Close the figure to release memory

##For loss

In [ ]:
from matplotlib import pyplot as plt
def print_history_loss(history,model_name):

  l = np.arange(len(history.history["loss"]))

  fig = plt.figure(dpi=150)
  # summarize history for accuracy
  # plt.plot(l, history.history['val_loss'])
  plt.plot(l,history.history['loss'])
  plt.title(f'model accuracy of {model_name}')
  plt.ylabel('Loss')
  plt.xlabel('epoch')
  plt.legend(['Loss','loss'])
  # Save the plot to the specified path
  plt.savefig(f'/content/drive/MyDrive/Model/photos/loss_{model_name}')

  plt.show()
  plt.close(fig)  # Close the figure to release memory

In [ ]:
#for accuracy


# For BanglaSER only
print_history_acc(ser_generalModel_history,"NeuroFusion")
print_history_acc(ser_CNN_model_history,"CovNetX")
print_history_acc(ser_hp_CNN_model_history,"CovNetM1")
print_history_acc(ser_best_CNN_model_history,"CascadeCovM1")
# print_history_acc(ser_LSTM_MODEL_history,"ser_LSTM_MODEL_history")
# print_history_acc(ser_general_cnn_model_history,"ser_general_cnn_model_history")
# print_history_acc(ser_CRNN_model_history,"ser_CRNN_model_history")
# print_history_acc(ser_general_LSTM_model_history,"ser_general_LSTM_model_history")
# print_history_acc(ser_another_CNN_model_history,"ser_another_CNN_model_history")

# # For mydataset only
# print_history_acc(my_generalModel_history,"my_generalModel_history")
# print_history_acc(my_CNN_model_history,"my_CNN_model_history")
# print_history_acc(my_hp_CNN_model_history,"my_hp_CNN_model_history")
# print_history_acc(my_best_CNN_model_history,"my_best_CNN_model_history")
# print_history_acc(my_LSTM_MODEL_history,"my_generalModel_history")
# print_history_acc(my_general_cnn_model_history,"my_CNN_model_history")
# print_history_acc(my_CRNN_model_history,"my_hp_CNN_model_history")
# print_history_acc(my_general_LSTM_model_history,"my_best_CNN_model_history")
# print_history_acc(my_another_CNN_model_history,"my_best_CNN_model_history")



# # For sust only
# print_history_acc(sust_generalModel_history ,"sust_generalModel_history ")
# print_history_acc(sust_CNN_model_history ,"sust_CNN_model_history ")
# print_history_acc(sust_hp_CNN_model_history ,"sust_hp_CNN_model_history ")
# print_history_acc(sust_best_CNN_model_history ,"sust_best_CNN_model_history ")


# print_history_acc(sust_generalModel_history, 'sust_generalModel')
# print_history_acc(sust_CNN_model_history, 'sust_CNN_model')
# print_history_acc(sust_hp_CNN_model_history, 'sust_hp_CNN_model')
# print_history_acc(sust_best_CNN_model_history, 'sust_best_CNN_model')
# print_history_acc(sust_LSTM_MODEL_history, 'sust_LSTM_MODEL')
# print_history_acc(sust_general_cnn_model_history, 'sust_general_cnn_model')
# print_history_acc(sust_CRNN_model_history, 'sust_CRNN_model')
# print_history_acc(sust_general_LSTM_model_history, 'sust_general_LSTM_model')
# print_history_acc(sust_another_CNN_model_history, 'sust_another_CNN_model')


#for loss


# For BanglaSER only
# print_history_loss(ser_generalModel_history,"ser_generalModel_history")
# print_history_loss(ser_CNN_model_history,"ser_CNN_model_history")
# print_history_loss(ser_hp_CNN_model_history,"ser_hp_CNN_model_history")
# print_history_loss(ser_best_CNN_model_history,"ser_best_CNN_model_history")
# print_history_loss(ser_LSTM_MODEL_history,"ser_LSTM_MODEL_history")
# print_history_loss(ser_general_cnn_model_history,"ser_general_cnn_model_history")
# print_history_loss(ser_CRNN_model_history,"ser_CRNN_model_history")
# print_history_loss(ser_general_LSTM_model_history,"ser_general_LSTM_model_history")
# print_history_loss(ser_another_CNN_model_history,"ser_another_CNN_model_history")


# # For mydataset only
# print_history_loss(my_generalModel_history,"my_generalModel_history")
# print_history_loss(my_CNN_model_history,"my_CNN_model_history")
# print_history_loss(my_hp_CNN_model_history,"my_hp_CNN_model_history")
# print_history_loss(my_best_CNN_model_history,"my_best_CNN_model_history")
# print_history_loss(my_LSTM_MODEL_history,"my_generalModel_history")
# print_history_loss(my_general_cnn_model_history,"my_CNN_model_history")
# print_history_loss(my_CRNN_model_history,"my_hp_CNN_model_history")
# print_history_loss(my_general_LSTM_model_history,"my_best_CNN_model_history")
# print_history_loss(my_another_CNN_model_history,"my_best_CNN_model_history")

# For Sust dataset
# print_history_loss(sust_generalModel_history, 'sust_generalModel')
# print_history_loss(sust_CNN_model_history, 'sust_CNN_model')
# print_history_loss(sust_hp_CNN_model_history, 'sust_hp_CNN_model')
# print_history_loss(sust_best_CNN_model_history, 'sust_best_CNN_model')
# print_history_loss(sust_LSTM_MODEL_history, 'sust_LSTM_MODEL')
# print_history_loss(sust_general_cnn_model_history, 'sust_general_cnn_model')
# print_history_loss(sust_CRNN_model_history, 'sust_CRNN_model')
# print_history_loss(sust_general_LSTM_model_history, 'sust_general_LSTM_model')
# print_history_loss(sust_another_CNN_model_history, 'sust_another_CNN_model')


print_history_loss(ser_generalModel_history,"NeuroFusion")
print_history_loss(ser_CNN_model_history,"CovNetX")
print_history_loss(ser_hp_CNN_model_history,"CovNetM1")
print_history_loss(ser_best_CNN_model_history,"CascadeCovM1")

In [ ]:

import csv

def find_accuracy(Model, model_name, X_test, y_test):
    # Evaluate the model on the test set
    csv_file = "/content/drive/MyDrive/Model/accuracy_files/results.csv"

    loss, sparse_categorical_accuracy = Model.evaluate(X_test, y_test)

    print(f'Result for {model_name} ')
    print('Loss:', loss)
    print('sparse_categorical_accuracy:', sparse_categorical_accuracy*100)
    print()

    # Save results to CSV file
    with open(csv_file, mode='a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow([model_name, loss, sparse_categorical_accuracy*100])


In [ ]:

# For BanglaSER only
# find_accuracy(ser_generalModel,"ser_generalModel",X_test,y_test)
# find_accuracy(ser_CNN_model,"ser_CNN_model",X_test,y_test)
# find_accuracy(ser_hp_CNN_model,"ser_hp_CNN_model",X_test,y_test)
# find_accuracy(ser_best_CNN_model,"ser_best_CNN_model",X_test,y_test)
# find_accuracy(ser_LSTM_MODEL,"ser_LSTM_MODEL",X_test,y_test)
# find_accuracy(ser_general_cnn_model,"ser_general_cnn_model",X_test,y_test)
# find_accuracy(ser_CRNN_model,"ser_CRNN_model",X_test,y_test)
# find_accuracy(ser_general_LSTM_model,"ser_general_LSTM_model",X_test,y_test)
# find_accuracy(ser_another_CNN_model,"ser_another_CNN_model",X_test,y_test)



# find_accuracy(my_generalModel,"my_generalModel",X_test1,y_test1)
# find_accuracy(my_CNN_model,"my_CNN_model",X_test1,y_test1)
# find_accuracy(my_hp_CNN_model,"my_hp_CNN_model",X_test1,y_test1)
# find_accuracy(my_best_CNN_model,"my_best_CNN_model",X_test1,y_test1)
# find_accuracy(my_generalModel,"my_generalModel",X_test1,y_test1)
# find_accuracy(my_CNN_model,"my_CNN_model",X_test1,y_test1)
# find_accuracy(my_hp_CNN_model,"my_hp_CNN_model",X_test1,y_test1)
# find_accuracy(my_best_CNN_model,"my_best_CNN_model",X_test1,y_test1)

# find_accuracy(model_with_hp,"generalModel_hp")
# find_accuracy(ser_generalModel,"ser_generalModel")
# find_accuracy(generalModel_rev,"generalModel_rev")
# find_accuracy(hp_CNN_model,"hp_CNN_model")
# find_accuracy(CNN_model,'CNN_model')
# find_accuracy(best_CNN_model,'best_CNN_model')
# find_accuracy(LSTM_MODEL,'LSTM_MODEL')
# find_accuracy(general_cnn_model,'general_cnn_model')
# find_accuracy(CRNN_model,'CRNN_model')
# find_accuracy(general_LSTM_model,'general_LSTM_model')
# find_accuracy(another_CNN_model,'another_CNN_model')

# find_accuracy(sust_generalModel, 'sust_generalModel',X_test2,y_test2)
# find_accuracy(sust_CNN_model, 'sust_CNN_model',X_test2,y_test2)
# find_accuracy(sust_hp_CNN_model, 'sust_hp_CNN_model',X_test2,y_test2)
# find_accuracy(sust_best_CNN_model, 'sust_best_CNN_model',X_test2,y_test2)
# find_accuracy(sust_another_CNN_model, 'sust_another_CNN_model',X_test2,y_test2)



find_accuracy(ser_generalModel,"NeuroFusion",X_test,y_test)
find_accuracy(ser_CNN_model,"CovNetX",X_test,y_test)
find_accuracy(ser_hp_CNN_model,"CovNetM1",X_test,y_test)
find_accuracy(ser_best_CNN_model,"CascadeCovM1",X_test,y_test)

In [ ]:
best_model_fit_history.evaluate

#K-Fold Cross Validation

In [ ]:
from tensorflow import expand_dims

from keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(monitor='sparse_categorical_accuracy', patience=20)

generalModel_history = generalModel.fit(expand_dims(X_train, axis=-1),y_train,epochs=500,validation_data=(X_test, y_test), callbacks=[early_stopping])


In [ ]:
# from keras.wrappers.scikit_learn import KerasClassifier
# from sklearn.model_selection import cross_val_score

# print(X.shape)
# print(Y.shape)
# CNN_Model = CNN_model(input_shape, num_classes)

# cnn_model = KerasClassifier(build_fn=create_CNN_model, epochs=10, batch_size=32, verbose=0)


# CNN_model.summary()
# cv_scores = cross_val_score(CNN_model, X, Y, cv=10)
# # Use cross_val_score with the defined model
# cv_scores = cross_val_score(CNN_model, X, Y, cv=10)
# print("Cross-validation scores:", cv_scores)
# print("Mean CV score:", cv_scores.mean())

In [ ]:
!pip install keras

In [ ]:
import numpy as np
from keras.wrappers.scikit_learn import KerasClassifier
from sklearn.model_selection import cross_val_score, KFold






def create_cnn_model():
    model = CNN_model(input_shape, num_classes)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model



In [ ]:
cnn_model = KerasClassifier(build_fn=create_cnn_model, epochs=200, batch_size=32, verbose=1)


In [ ]:
from keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)


In [ ]:
n_folds = 5  # You can choose any number of folds you want

kfold = KFold(n_splits=n_folds, shuffle=True, random_state=42)
cv_scores = cross_val_score(cnn_model, X,Y, cv=kfold, fit_params={'callbacks': [early_stopping]})


In [ ]:
print("Cross-validation scores:", cv_scores)
print("Mean CV score:", cv_scores.mean())


###Graphs

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from keras.callbacks import EarlyStopping
import numpy as np

# Define the number of folds
n_folds = 5

# Create KFold object
kfold = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# List to store accuracy for each fold
fold_accuracies = []

# Loop through each fold
for train_idx, val_idx in kfold.split(X, Y):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = Y[train_idx], Y[val_idx]

    # Create and compile your model
    model = create_cnn_model()
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # Define EarlyStopping callback
    early_stopping = EarlyStopping(monitor='accuracy', patience=10, restore_best_weights=True)

    # Train the model using the training data and validate on the validation data
    history = model.fit(X_train, y_train, epochs=200, batch_size=32, validation_data=(X_val, y_val),
                        callbacks=[early_stopping],verbose=False)

    # Plot training history
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.legend()
    plt.title('Training History')
    plt.xlabel('Epoch')
    plt.ylabel('Value')
    plt.show()

    # Evaluate the model on the validation data
    val_preds = model.predict(X_val)
    val_preds_classes = np.argmax(val_preds, axis=1)
    fold_accuracy = accuracy_score(y_val, val_preds_classes)
    fold_accuracies.append(fold_accuracy)

# Calculate and print the final mean accuracy
mean_accuracy = np.mean(fold_accuracies)
print("Final Mean Accuracy:", mean_accuracy)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from keras.callbacks import EarlyStopping
import numpy as np

# Define the number of folds
n_folds = 10

# Create KFold object
kfold = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Create subplots for loss and accuracy
fig, axs = plt.subplots(n_folds, 2, figsize=(12, 4*n_folds))
axs = axs.flatten()

# Loop through each fold
for i, (train_idx, val_idx) in enumerate(kfold.split(X, Y)):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = Y[train_idx], Y[val_idx]

    # Create and compile your model
    model = create_cnn_model()
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # Define EarlyStopping callback
    early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    # Train the model using the training data and validate on the validation data
    history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_val, y_val),
                        callbacks=[early_stopping])

    # Plot loss and accuracy for this fold
    axs[i].plot(history.history['loss'], label='Train Loss')
    axs[i].plot(history.history['val_loss'], label='Validation Loss')
    axs[i].plot(history.history['accuracy'], label='Train Accuracy')
    axs[i].plot(history.history['val_accuracy'], label='Validation Accuracy')
    axs[i].set_title(f'Fold {i+1} Training History')
    axs[i].set_xlabel('Epoch')
    axs[i].set_ylabel('Value')
    axs[i].legend()

# Adjust spacing between subplots and display
plt.tight_layout()
plt.show()


##Save model to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Assuming you have a trained model called 'model'
model.save('/content/drive/MyDrive/BanglaSpeechData/Dataset/my_model_silent.h5')


#Some Useful Graphs

##Preprocessing for graphs

In [ ]:
from keras.models import load_model

model_path = '/content/drive/MyDrive/BanglaSpeechData/Dataset/my_model_silent.h5';
model = load_model(model_path)


In [ ]:
print(X_test)

In [ ]:
Y_pred = {}

def find_yPred(Model,modelName):
  # Assuming you have already loaded your model and test data
  # Make predictions for the test data
  Y_pred[modelName] = np.array(Model.predict(X_test))

In [ ]:

find_yPred(ser_generalModel,'NeuroFusion')
find_yPred(ser_CNN_model,'CovNetX')
find_yPred(ser_hp_CNN_model,'CovNetM1')
find_yPred(ser_best_CNN_model,'CascadeCovM1')

In [ ]:
final_data = {}

In [ ]:
def preprocess_for_graphs(modelName):
  # Convert the array of class probabilities to the corresponding emotion labels
  predicted_emotion_index = np.argmax(Y_pred[modelName], axis=1)
  emotion_labels = ['Happy', 'Sad', 'Angry', 'Surprised', 'Neutral', 'Disgust', 'Fear']
  pred = np.array([emotion_labels[index] for index in predicted_emotion_index])
  actual = np.array([emotion_labels[index] for index in y_test])



  def calculate_accuracy(pred, actual):
      correct_predictions = sum(pred == actual)
      total_samples = len(actual)
      accuracy = correct_predictions / total_samples * 100
      return accuracy

  # import numpy as np

  # Assuming pred and actual are arrays containing the predicted and actual labels
  pred = np.array(pred)  # Replace with your predicted labels
  actual = np.array(actual) # Replace with your actual labels

  accuracy = calculate_accuracy(pred, actual)
  print(f"Accuracy: {accuracy:.2f}%")


  # Stack the arrays side by side
  comparison = np.column_stack((actual, pred))

  print('  Actual------Pred\n')
  print(comparison)
  final_data[modelName] = {"actual":actual,"pred":pred}


In [ ]:
preprocess_for_graphs("NeuroFusion")
preprocess_for_graphs('CovNetX')
preprocess_for_graphs('CovNetM1')
preprocess_for_graphs('CascadeCovM1')

##Confusion Matrix

In [ ]:
def make_confusion_matrix(modelName):
  import numpy as np
  from sklearn.metrics import confusion_matrix
  import seaborn as sns
  import matplotlib.pyplot as plt


  # Get the unique classes (labels) present in the data
  classes = np.unique(np.concatenate((final_data[modelName]["pred"], final_data[modelName]["actual"])))

  # Create the confusion matrix
  cm = confusion_matrix(final_data[modelName]["actual"], final_data[modelName]["pred"], labels=classes)

  # Create a heatmap to visualize the confusion matrix
  plt.figure(figsize=(8, 6))
  sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
  plt.xlabel('Predicted')
  plt.ylabel('Actual')
  plt.title(f'Confusion Matrix of {modelName}')
  plt.show()

In [ ]:
make_confusion_matrix("NeuroFusion")
make_confusion_matrix('CovNetX')
make_confusion_matrix('CovNetM1')
make_confusion_matrix('CascadeCovM1')

# final_data['generalModel']['pred']

##Bar Plot

In [ ]:
def show_bar_chart(modelName):
  import matplotlib.pyplot as plt

  # Count the occurrences of each class in the actual array
  class_counts = np.unique(final_data[modelName]["pred"], return_counts=True)

  # Create a bar plot
  plt.figure(figsize=(8, 6))
  plt.bar(class_counts[0], class_counts[1])
  plt.xlabel('Class')
  plt.ylabel('Count')
  plt.title(f'Class Distribution of {modelName}')
  plt.show()

In [ ]:
show_bar_chart("NeuroFusion")
show_bar_chart('CovNetX')
show_bar_chart('CovNetM1')
show_bar_chart('CascadeCovM1')
# show_bar_chart('general_cnn_model')
# show_bar_chart('CRNN_model')
# show_bar_chart('general_LSTM_model')
# show_bar_chart('another_CNN_model')

##Multi Class ROC Curve

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np

def show_ROC_Curve(modelName):

  # Get the unique classes
  classes = np.unique(np.concatenate((final_data[modelName]["pred"], final_data[modelName]["actual"])))

  # Convert class labels to binary format for each class
  binary_preds = np.array([np.where(classes == p, 1, 0) for p in final_data[modelName]["pred"]])
  binary_actuals = np.array([np.where(classes == a, 1, 0) for a in final_data[modelName]["actual"]])

  # Initialize dictionaries to store fpr and tpr for each class
  fpr_dict = {}
  tpr_dict = {}

  # Calculate ROC curve for each class
  for i, c in enumerate(classes):
      fpr, tpr, _ = roc_curve(binary_actuals[:, i], binary_preds[:, i])
      fpr_dict[c] = fpr
      tpr_dict[c] = tpr

  # Plot the ROC curves for each class
  plt.figure(figsize=(8, 6))
  for c in classes:
      plt.plot(fpr_dict[c], tpr_dict[c], lw=2, label='ROC curve for %s' % c)

  plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
  plt.xlim([0.0, 1.0])
  plt.ylim([0.0, 1.05])
  plt.xlabel('False Positive Rate')
  plt.ylabel('True Positive Rate')
  plt.title(f'Multi-class Receiver Operating Characteristic (ROC) Curve of {modelName}')
  plt.legend(loc='lower right')
  plt.show()


In [ ]:
show_ROC_Curve("generalModel")
show_ROC_Curve('CNN_model')
show_ROC_Curve('best_CNN_model')
show_ROC_Curve('LSTM_MODEL')
show_ROC_Curve('general_cnn_model')
show_ROC_Curve('CRNN_model')
show_ROC_Curve('general_LSTM_model')
show_ROC_Curve('another_CNN_model')

##Precision-Recall Curve

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.utils.multiclass import unique_labels

def show_precision_recall_curve(modelName):
  # Get the unique classes
  classes = np.unique(np.concatenate((final_data[modelName]["pred"], final_data[modelName]["actual"])))

  # Convert class labels to binary format for each class
  binary_preds = np.array([np.where(classes == p, 1, 0) for p in final_data[modelName]["pred"]])
  binary_actuals = np.array([np.where(classes == a, 1, 0) for a in final_data[modelName]["actual"]])

  # Calculate precision-recall curve for each class
  precision = dict()
  recall = dict()
  average_precision = dict()
  for i, c in enumerate(classes):
      precision[c], recall[c], _ = precision_recall_curve(binary_actuals[:, i], binary_preds[:, i])
      average_precision[c] = average_precision_score(binary_actuals[:, i], binary_preds[:, i])

  # Plot the Precision-Recall Curve for each class
  plt.figure(figsize=(8, 6))
  for c in classes:
      plt.plot(recall[c], precision[c], lw=2, label='Precision-Recall curve for %s (AP = %0.2f)' % (c, average_precision[c]))

  plt.xlim([0.0, 1.0])
  plt.ylim([0.0, 1.05])
  plt.xlabel('Recall')
  plt.ylabel('Precision')
  plt.title(f'Precision-Recall Curve of {modelName}')
  plt.legend(loc='upper right')
  plt.show()


In [ ]:
show_precision_recall_curve("generalModel")
show_precision_recall_curve('CNN_model')
show_precision_recall_curve('best_CNN_model')
show_precision_recall_curve('LSTM_MODEL')
show_precision_recall_curve('general_cnn_model')
show_precision_recall_curve('CRNN_model')
show_precision_recall_curve('general_LSTM_model')
show_precision_recall_curve('another_CNN_model')

##MultiClass Confusion Matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.utils.multiclass import unique_labels

# Example predicted and actual arrays for multi-class classification (5 classes)
# pred = np.array(['happy', 'angry', 'sad', 'happy', 'neutral', 'happy', 'sad', 'neutral'])
# actual = np.array(['happy', 'angry', 'sad', 'happy', 'angry', 'happy', 'sad', 'happy'])

def show_multiclass_confusion_matrix(modelName):
  # Get the unique classes
  classes = np.unique(np.concatenate((final_data[modelName]["pred"], final_data[modelName]["actual"])))

  # Create the confusion matrix
  cm = confusion_matrix(final_data[modelName]["actual"], final_data[modelName]["pred"], labels=classes)

  # Create a heatmap for the confusion matrix
  plt.figure(figsize=(8, 6))
  sns.heatmap(cm, annot=True, cmap='Blues', xticklabels=classes, yticklabels=classes)
  plt.xlabel('Predicted')
  plt.ylabel('Actual')
  plt.title(f'Multiclass Confusion Matrix {modelName}')
  plt.show()


In [ ]:
show_multiclass_confusion_matrix("generalModel")
show_multiclass_confusion_matrix('CNN_model')
show_multiclass_confusion_matrix('best_CNN_model')
show_multiclass_confusion_matrix('LSTM_MODEL')
show_multiclass_confusion_matrix('general_cnn_model')
show_multiclass_confusion_matrix('CRNN_model')
show_multiclass_confusion_matrix('general_LSTM_model')
show_multiclass_confusion_matrix('another_CNN_model')

##Class-wise Precision-Recall Curves

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.utils.multiclass import unique_labels

# Example predicted and actual arrays for multi-class classification (5 classes)
# pred = np.array(['happy', 'angry', 'sad', 'happy', 'neutral', 'happy', 'sad', 'neutral'])
# actual = np.array(['happy', 'angry', 'sad', 'happy', 'angry', 'happy', 'sad', 'happy'])

# Get the unique classes
classes = np.unique(np.concatenate((pred, actual)))

# Convert class labels to binary format for each class
binary_preds = np.array([np.where(classes == p, 1, 0) for p in pred])
binary_actuals = np.array([np.where(classes == a, 1, 0) for a in actual])

# Calculate precision-recall curve for each class
for i, c in enumerate(classes):
    precision, recall, _ = precision_recall_curve(binary_actuals[:, i], binary_preds[:, i])
    average_precision = average_precision_score(binary_actuals[:, i], binary_preds[:, i])

    # Plot the Precision-Recall Curve for each class
    plt.figure(figsize=(8, 6))
    plt.step(recall, precision, lw=2, label='Precision-Recall curve for %s (AP = %0.2f)' % (c, average_precision))
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve for Class: %s' % c)
    plt.legend(loc='upper right')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.show()


##Cumulative Gain Curve

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
from sklearn.utils.multiclass import unique_labels

# Example predicted and actual arrays for multi-class classification (5 classes)
# pred = np.array(['happy', 'angry', 'sad', 'happy', 'neutral', 'happy', 'sad', 'neutral'])
# actual = np.array(['happy', 'angry', 'sad', 'happy', 'angry', 'happy', 'sad', 'happy'])

# Get the unique classes
classes = np.unique(np.concatenate((pred, actual)))

# Convert class labels to binary format for each class
binary_preds = np.array([np.where(classes == p, 1, 0) for p in pred])
binary_actuals = np.array([np.where(classes == a, 1, 0) for a in actual])

# Calculate cumulative gain curve for each class
def cumulative_gain_curve(y_true, y_probs, positive_class=1):
    idx_sorted = np.argsort(y_probs[:, positive_class])[::-1]
    y_true_sorted = y_true[idx_sorted][:, positive_class]
    gains = np.cumsum(y_true_sorted) / np.sum(y_true_sorted)
    gains = np.append([0], gains)
    return gains

for i, c in enumerate(classes):
    gains = cumulative_gain_curve(binary_actuals, binary_preds)
    # Plot the Cumulative Gain Curve for each class
    plt.plot(np.linspace(0, 1, len(gains)), gains, lw=2, label='Cumulative Gain curve for %s' % c)

plt.xlabel('Percentage of dataset')
plt.ylabel('Cumulative Gain')
plt.title('Cumulative Gain Curve')
plt.legend(loc='lower right')
plt.show()


##Lift Curve

In [ ]:
def lift_curve(y_true, y_probs, positive_class=1):
    gains = cumulative_gain_curve(y_true, y_probs, positive_class)
    random_gains = np.linspace(0, 1, len(gains))
    lift = gains / random_gains
    return lift

for i, c in enumerate(classes):
    lift = lift_curve(binary_actuals, binary_preds)
    # Plot the Lift Curve for each class
    plt.plot(np.linspace(0, 1, len(lift)), lift, lw=2, label='Lift curve for %s' % c)

plt.xlabel('Percentage of dataset')
plt.ylabel('Lift')
plt.title('Lift Curve')
plt.legend(loc='lower right')
plt.show()


# Make Everything Automate  

In [ ]:

from tensorflow import expand_dims

from keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='sparse_categorical_accuracy', patience=20)

accs = []
# compile the model with different optimizers
optimizer_list = [keras.optimizers.SGD(),
                  keras.optimizers.RMSprop(),
                  keras.optimizers.Adagrad(),
                  keras.optimizers.Adadelta(),
                  keras.optimizers.Adam()]
for optimizer in optimizer_list:

    newModel = LSTM_MODEL(5)

    newModel.compile(
        optimizer=optimizer,  # Optimizer
        # Loss function to minimize
        loss=keras.losses.SparseCategoricalCrossentropy(),
        # List of metrics to monitor
        metrics=[keras.metrics.SparseCategoricalAccuracy()],
    )

    # import tensorflow as tf

    from tensorflow import expand_dims



    # history = newModel.fit(expand_dims(X_train, axis=-1),y_train,epochs=300,verbose=None)
    history = newModel.fit(expand_dims(X_train, axis=-1),y_train,epochs=500, callbacks=[early_stopping],verbose=None)

    # Evaluate the newModel on the test set
    loss, accuracy = newModel.evaluate(X_test, y_test)
    accs.append(accuracy*100)
    print(f'----------------for {optimizer}------------------- ')
    print('Loss:', loss*100)
    print('Accuracy:', accuracy*100)

#finding Appropiate Epoch for our dataset

In [ ]:
from tensorflow import expand_dims
epoch_with_accuracy= {

}
# compile the model with different optimizers
epoche = [100,150,200,250,300,350,400]

for epoch in epoche:
    model.compile(
        optimizer=keras.optimizers.Adam(),  # Optimizer
        # Loss function to minimize
        loss=keras.losses.SparseCategoricalCrossentropy(),
        # List of metrics to monitor
        metrics=[keras.metrics.SparseCategoricalAccuracy()],
    )


    history = model.fit(expand_dims(X_train, axis=-1),y_train,epochs=epoch,verbose=None)


    # Evaluate the model on the test set
    loss, accuracy = model.evaluate(X_test, y_test)

    epoch_with_accuracy[epoch] = accuracy*100

    print(f'----------------for {epoch}------------------- ')
    print('Loss:', loss*100)
    print('Accuracy:', accuracy*100)

In [ ]:
acc = pd.DataFrame(epoch_with_accuracy,index=[0])

acc.head()

#Testing sector

In [ ]:
# Convert X_train to a DataFrame
df_train = pd.DataFrame(X_train)

In [ ]:
from pivottablejs import pivot_ui
pivot_ui(df_train)

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, LSTM, Dense, Dropout
from keras.optimizers import Adadelta


height,width = 28,180

# Load and preprocess the data
# Assuming you have X_train, X_test, y_train, y_test from your feature extraction step

# # Convert the features and labels to NumPy arrays
X_train = np.array(X_train)
X_test = np.array(X_test)
y_train = np.array(y_train)
y_test = np.array(y_test)

# # Encode the categorical labels into numerical values
# le = LabelEncoder()
# y_train = le.fit_transform(y_train)
# y_test = le.transform(y_test)

# Reshape the input features depending on the model architecture
# For CNN models, assuming the features are in the form of spectrograms
# Reshape to (samples, height, width, channels)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], X_train.shape[2], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], X_test.shape[2], 1)

# Split the data into training and validation sets
X_trains, X_val, y_trains, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# CNN Model
cnn_model = Sequential()
cnn_model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(height, width, 1)))
cnn_model.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model.add(Flatten())
cnn_model.add(Dense(128, activation='relu'))
cnn_model.add(Dense(num_classes, activation='softmax'))

cnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn_model.fit(X_trains, y_trains, validation_data=(X_val, y_val), epochs=10, batch_size=32)

# LSTM Model
lstm_model = Sequential()
lstm_model.add(LSTM(128, return_sequences=True, input_shape=(timesteps, input_dim)))
lstm_model.add(LSTM(128))
lstm_model.add(Dense(num_classes, activation='softmax'))

lstm_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
lstm_model.fit(X_trains, y_trains, validation_data=(X_val, y_val), epochs=10, batch_size=32)

# Hybrid Model (CNN + LSTM)
hybrid_model = Sequential()
hybrid_model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(height, width, 1)))
hybrid_model.add(MaxPooling2D(pool_size=(2, 2)))
hybrid_model.add(Flatten())
hybrid_model.add(Reshape((new_height, new_width)))
hybrid_model.add(LSTM(128))
hybrid_model.add(Dense(num_classes, activation='softmax'))

hybrid_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hybrid_model.fit(X_trains, y_trains, validation_data=(X_val, y_val), epochs=10, batch_size=32)

# Evaluate the models on the test set
cnn_model.evaluate(X_test, y_test)
lstm_model.evaluate(X_test, y_test)
hybrid_model.evaluate(X_test, y_test)


In [ ]:
from sklearn.neural_network import MLPClassifier
#Initialize the Multi Layer Perceptron Classifier
model=MLPClassifier(alpha=0.01, batch_size=5, epsilon=1e-08, hidden_layer_sizes=(1200,), learning_rate='adaptive', max_iter=500)

In [ ]:
model.fit(X_train,y_train)

In [ ]:
model.save('/content/drive/MyDrive/MachineLearningModel/model')



In [ ]:
!pip install joblib

In [ ]:
import joblib

joblib.dump(model,'/content/drive/MyDrive/MachineLearningModel/model.pkl')


In [ ]:
import pickle
# Writing different model files to file
with open( 'modelForPrediction1.sav', 'wb') as f:
    pickle.dump(model,f)

In [ ]:
filename = 'modelForPrediction1.sav'
loaded_model = pickle.load(open(filename, 'rb')) # loading the model file from the storage

feature=extract_feature("/content/drive/MyDrive/Colab_Notebooks/RAVDESS_Emotional_speech_audio/speech-emotion-recognition-ravdess-data/Actor_01/03-01-01-01-01-01-01.wav", mfcc=True, chroma=True, mel=True)

feature=feature.reshape(1,-1)

prediction=loaded_model.predict(feature)
prediction

In [ ]:
pwd

In [ ]:
# Assuming your LSTM model is named "model"
model.save('LSTM_MODEL_2.h5')

#Transfer learning VGGish model

In [ ]:
!pip install tensorflow tensorflow-hub tensorflow-io


In [ ]:
import tensorflow_hub as hub

vggish_model = hub.load("https://tfhub.dev/google/vggish/1")



In [ ]:
import tensorflow as tf
import tensorflow_io as tfio

def preprocess_audio(file_path):
    audio = tf.io.read_file(file_path)
    audio, _ = tf.audio.decode_wav(audio)
    audio = tfio.audio.resample(audio, 16000, 16000)
    return audio


In [ ]:
def extract_embeddings(audio):
    embeddings = vggish_model(audio)
    return embeddings


In [ ]:
!pip install keras-tuner

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam
from kerastuner.tuners import RandomSearch
from keras.optimizers import RMSprop
import keras

num_classes=10

# Define the model-building function
def build_model(hp):
    model = Sequential()

    # First convolutional layer
    model.add(Conv1D(filters=X_train.shape[1], kernel_size=3, activation='relu', input_shape=(X_train.shape[1], 1)))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.2))

    # Tune the number of convolutional layers and their parameters
    num_conv_layers = hp.Int('num_conv_layers', min_value=1, max_value=3, default=2)
    for i in range(num_conv_layers):
        filters = hp.Int('filters_' + str(i), min_value=32, max_value=256, step=32)
        kernel_size = hp.Choice('kernel_size_' + str(i), values=[3, 5])
        model.add(Conv1D(filters=filters, kernel_size=kernel_size, activation='relu'))
        model.add(MaxPooling1D(pool_size=2))
        model.add(Dropout(0.2))

    model.add(Flatten())

    # Tune the number of dense layers and their units
    num_dense_layers = hp.Int('num_dense_layers', min_value=1, max_value=3, default=2)
    for i in range(num_dense_layers):
        units = hp.Int('units_' + str(i), min_value=64, max_value=512, step=64)
        model.add(Dense(units, activation='relu'))
        model.add(Dropout(0.3))

    model.add(Dense(num_classes, activation='softmax'))


    # Set the learning rate
    learning_rate = 0.001

    # Optionally, you can also set other hyperparameters like rho (decay factor) and epsilon
    rho = 0.9
    epsilon = 1e-08

    # Compile the model with RMSprop optimizer
    optimizer = RMSprop(learning_rate=learning_rate, rho=rho, epsilon=epsilon)

    # Loss function
    loss = keras.losses.SparseCategoricalCrossentropy()

    # Compile the model with the specified optimizer, loss function, and metrics
    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=[
            keras.metrics.SparseCategoricalAccuracy(),
            keras.metrics.CategoricalAccuracy(),
            # keras.metrics.BinaryAccuracy()
        ]
    )
    return model

# Load and preprocess your data (replace with your data)
X_train = np.random.random((1000, 100))
y_train = keras.utils.to_categorical(np.random.randint(num_classes, size=(1000,)), num_classes=num_classes)

# Initialize the RandomSearch tuner
tuner = RandomSearch(
    build_model,
    objective='val_categorical_accuracy',
    max_trials=10,  # Number of trials
    directory='tuner_directory',  # Directory for tuner logs
    project_name='cnn_tuner'  # Name of the project
)

# Search for the best hyperparameters
tuner.search(X_train, y_train, epochs=10, validation_split=0.2)

# Get the best hyperparameters and build the final model
best_model = tuner.get_best_models(num_models=1)[0]
best_hyperparameters = tuner.get_best_hyperparameters(num_trials=1)[0]

# Print the best hyperparameters
print("Best hyperparameters:", best_hyperparameters)

# Print the summary of the best model
best_model.summary()


In [ ]:
from keras.models import Sequential
from keras.layers import Conv1D, MaxPooling1D, Dense, Flatten, Dropout
from keras.optimizers import Adam
from keras.losses import CategoricalCrossentropy
from keras.metrics import CategoricalAccuracy


def best_CNN_model(hp):


  model = Sequential()

  # First convolutional layer
  model.add(Conv1D(filters=X_train.shape[1], kernel_size=3, activation='relu', input_shape=(X_train.shape[1], 1)))
  model.add(MaxPooling1D(pool_size=2))
  model.add(Dropout(0.2))

  # Second convolutional layer
  model.add(Conv1D(filters=X_train.shape[1], kernel_size=3, activation='relu'))
  model.add(MaxPooling1D(pool_size=2))
  model.add(Dropout(0.2))

  # Third convolutional layer
  model.add(Conv1D(filters=360, kernel_size=3, activation='relu'))
  model.add(MaxPooling1D(pool_size=2))
  model.add(Dropout(0.2))

  # Flatten the output from the previous layer
  model.add(Flatten())

  # Fully connected layers
  model.add(Dense(720, activation='relu'))
  model.add(Dropout(0.3))
  model.add(Dense(360, activation='relu'))
  model.add(Dropout(0.3))
  model.add(Dense(180, activation='relu'))
  model.add(Dropout(0.3))
  model.add(Dense(90, activation='relu'))

  # Output layer
  model.add(Dense(n, activation='softmax'))
  return model


best_CNN_model = best_CNN_model(num_classes)

best_CNN_model.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(),
        keras.metrics.CategoricalAccuracy()
    ]
)

# Print the model summary
best_CNN_model.summary()


In [ ]:
print(X_train.shape)
print(X_test.shape)


In [ ]:
print(y_train.shape)
print(y_test.shape)

#Try Transfer Learning Model


In [ ]:
X_train.shape

In [ ]:
base_model = keras.applications.VGG19(include_top=False,
    input_shape=(32,180,3),
    classifier_activation="softmax",)

for layer in base_model.layers:
    layer.trainable = False

In [ ]:
model = Sequential()

model.add(base_model)
model.add(Flatten())
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.2))

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.2))

model.add(Dense(56, activation='relu'))
model.add(Dropout(0.2))

model.add(Dense(num_classes, activation='softmax'))  # num_classes is the number of classes in your sign language dataset

model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'],run_eagerly=True)

In [ ]:
model.summary()

In [ ]:
X_train.shape

In [ ]:
from tensorflow import expand_dims

expand_dims(expand_dims(X_train, axis=-1), axis=-1).shape

In [ ]:
# Reshape myArr to have shape (3, 1, 180, 1)
X_train_ext = X_train.reshape(X_train.shape[0], 1, X_train.shape[1], 1)

# Repeat X_train along the second axis to have 32 copies
arr = np.tile(X_train_ext, (1, 32, 1, 3))

In [ ]:
y_train.shape

In [ ]:
myArr = np.array([1, 0, 1,1,1,1,1,1,1,1,1,1,0])

# Reshape myArr to match the desired shape
reshaped_arr = y_train.reshape((y_train.shape[0], 1, 1, 1))

# Use numpy.tile to repeat the values along the necessary dimensions
expected_arr = np.tile(reshaped_arr, (1, 1, 5, 2))

print(expected_arr.shape)

In [ ]:
expected_arr = np.tile(y_train.reshape(-1, 1), (1, 2))

In [ ]:
expected_arr.shape

In [ ]:
history = model.fit(arr, expected_arr, epochs=100)